Teste base de correção de anotação de enfermagem com Gemini.

In [ ]:
import google.generativeai as genai
import pandas as pd
import os 


# Api Gemini
os.environ["GOOGLE_API_KEY"] = "Sua-chave-de-API-aqui"
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# Inicializa o modelo
model = genai.GenerativeModel('models/gemini-3-flash-preview')

In [7]:
import PyPDF2

def extrair_texto_pdf(caminho_do_pdf):
    texto = "Enviando por email anotacao-de-enfermagem.pdf"
    with open(caminho_do_pdf, "rb") as arquivo:
        leitor = PyPDF2.PdfReader(arquivo)
        # Vamos ler as primeiras páginas para não estourar o limite inicial
        for pagina in leitor.pages:
            texto += pagina.extract_text()
    return texto

contexto_cofen = extrair_texto_pdf("Enviando por email anotacao-de-enfermagem.pdf")

f=(f-string) formatted string
3000 caracteres do PDF do COFEN

In [10]:
def sugerir_com_base_no_pdf(anotacao_original, contexto_cofen, cargo):
    prompt = f"""
    Normas Técnicas: {contexto_cofen[:3000]}
    
    Perfil do Usuário: {"TÉCNICO DE ENFERMAGEM" if cargo == '1' else "ENFERMEIRO"}

    Sua tarefa é revisar o registro de saúde. Aja como um revisor de prontuário de enfermagem. Sua tarefa é transformar anotações informais ou com erros em registros técnicos e legais.
    
    REGRAS DE OURO:
    1. Se for TÉCNICO: Use termos de ANOTAÇÃO (sinais observáveis, queixas). Proibido termos de diagnóstico ou exame físico (ex: 'ascite', 'globo vesical', 'murmúrio vesicular').
    2. Se for ENFERMEIRO: Use termos de EVOLUÇÃO (exame físico completo e termos propedêuticos).
    
    ESTRUTURA:
    - Sugestão: [Texto]
    - Por que: [ Breve justificativa técnica em uma linha]
    REGRAS RÍGIDAS:
    - Se a informação faltar (ex: lateralidade), use colchetes como [especificar lado].
    - Não use frases como "Aqui está a análise" ou "Com base nas normas".
    - Seja direto e técnico.
    - Não mencionar assinatura, carimbo ou data.


    Anotação: "{anotacao_original}"
    """
    
    response = model.generate_content(prompt)
    return response.text.strip()
    
def responder_duvida_enfermagem(pergunta, contexto_cofen):
    prompt = f"""
    Contexto Normativo: {contexto_cofen[:5000]}
    Você é um Consultor Técnico de Enfermagem especialista em Legislação e Auditoria.
    seja direto e tecnico, não dê introduções ou explicações desnecessárias, responda apenas o que foi perguntado, com base nas normas do COFEN.
    Responda à seguinte dúvida de forma profissional, direta e baseada nas normas:
    Pergunta: "{pergunta}"
    """
    response = model.generate_content(prompt)
    return response.text.strip()

# Testeando o sistema

print("Simulador de Auditoria de Enfermagem")

while True:  
    print("\n IDENTIFICAÇÃO")
    print("[1] Técnico/Auxiliar")
    print("[2] Enfermeiro")
    print("[0] Sair")
    cargo = input("Selecione seu cargo: ")
    
    if cargo == '0': 
        print("Encerrando o sistema...")
        break
    if cargo not in ['1', '2']: 
        print("Opção inválida!")
        continue

    sair_geral = False # Variável de controle para sair de vez
    
    while True:  # Loop Secundário (Ações)
        print(f"\nMODO: {'Técnico' if cargo == '1' else 'Enfermeiro'}")
        print("[1] Corrigir Anotação")
        print("[2] Tirar Dúvida Técnica")
        print("[9] Voltar/Trocar Cargo")
        print("[0] Sair")
        
        opcao = input("Escolha uma opção: ")

        if opcao == '0':
            print("Encerrando o sistema...")
            sair_geral = True
            break # Sai do loop de ações
        
        elif opcao == '9':
            break # Apenas volta para a IDENTIFICAÇÃO
            
        elif opcao == '1':
            texto = input("Digite a anotação (ou '0' para sair): ")
            if texto == '0':
                sair_geral = True
                break
            print("\n" + "="*30)
            print(sugerir_com_base_no_pdf(texto, contexto_cofen, cargo))
            print("="*30)
        
        elif opcao == '2':
            pergunta = input("Qual sua dúvida (ou '0' para sair)? ")
            if pergunta == '0':
                sair_geral = True
                break
            print("\n" + "="*30)
            print(responder_duvida_enfermagem(pergunta, contexto_cofen))
            print("="*30)
        
        else:
            print("Opção inválida!")

    if sair_geral: # Se o usuário pediu para sair lá dentro, quebra o loop principal também
        break

Simulador de Auditoria de Enfermagem

 IDENTIFICAÇÃO
[1] Técnico/Auxiliar
[2] Enfermeiro
[0] Sair

MODO: Técnico
[1] Corrigir Anotação
[2] Tirar Dúvida Técnica
[9] Voltar/Trocar Cargo
[0] Sair
Encerrando o sistema...
